# Screen Capture Tests

This notebook tests the dialoghelper **capture** functionality, which allows taking screenshots from Python notebook code.

## How it works

1. `setup_share()` — Injects `screenshot.js` into the browser (registers event listeners)
2. `start_share()` — Triggers the browser's screen/window picker dialog
3. `capture_screen()` — Captures the current frame as a PIL Image

The flow uses Jupyter's bidirectional data transfer:
- Python fires a `captureScreen` event → JS grabs a frame from the video track → `pushData()` sends the base64 image back to Python via `/push_data_blocking_`

**Run each cell in order.**

In [ ]:
# Cell 1: Setup & Imports
from dialoghelper import dh_settings
dh_settings['port'] = 8000

from dialoghelper.capture import setup_share, start_share, capture_screen, capture_tool
import PIL.Image

print('Capture imports ready!')
print(f'Port: {dh_settings["port"]}')

In [ ]:
# Cell 2: Setup screen sharing
# This injects screenshot.js into the browser, which registers:
# - 'shareScreen' event listener (opens screen picker)
# - 'captureScreen' event listener (grabs frame + pushData)
setup_share()
print('Screen sharing setup complete! Event listeners registered.')

In [ ]:
# Cell 3: Start screen sharing
# This triggers the browser's screen/window picker dialog.
# Select a window or screen to share, then run the next cell.
start_share()
print('Screen picker triggered! Select a window or screen to share.')

In [ ]:
# Cell 4: Capture a screenshot
# This fires the captureScreen event, grabs the current frame,
# and returns it as a PIL Image.
img = await capture_screen()
print(f'Captured image: {img.size[0]}x{img.size[1]} pixels, mode={img.mode}')

# Display the image inline
img.thumbnail((800, 600))  # Resize for display
img

In [ ]:
# Cell 5: Test capture_tool (LLM-friendly wrapper)
# capture_tool is decorated with @llmtool for use as an LLM tool.
# It returns PIL Image on success, or error string on failure.
result = await capture_tool()
if isinstance(result, PIL.Image.Image):
    print(f'capture_tool succeeded: {result.size[0]}x{result.size[1]}')
    result.thumbnail((800, 600))
    display(result)
else:
    print(f'capture_tool returned: {result}')

In [ ]:
# Cell 6: Multiple captures
# Each call to capture_screen() grabs the latest frame.
# Switch windows between captures to see different content.
import time

for i in range(3):
    img = await capture_screen()
    print(f'Capture {i+1}: {img.size[0]}x{img.size[1]}')
    if i < 2:
        print('  Waiting 2 seconds before next capture...')
        time.sleep(2)

print('\nAll captures complete! The last image:')
img.thumbnail((800, 600))
img

In [ ]:
# Cell 7: Save a screenshot to file
import tempfile, os

img = await capture_screen()
path = os.path.join(tempfile.gettempdir(), 'dialeng_capture.png')
img.save(path)
print(f'Screenshot saved to: {path}')
print(f'File size: {os.path.getsize(path):,} bytes')

## Summary

If all cells ran successfully, you've verified:

- **setup_share()** — Injects screenshot.js event listeners into the browser
- **start_share()** — Triggers browser screen/window picker dialog
- **capture_screen()** — Captures current frame as PIL Image (async)
- **capture_tool()** — LLM-friendly wrapper (async, returns Image or error string)

### Architecture

```
Python: capture_screen()  →  event_get_a('captureScreen')
                          →  fire_event_a('captureScreen', {idx: uuid})
                          →  WebSocket broadcast to browser

Browser: captureScreen listener
         → ImageCapture.grabFrame()
         → canvas.toDataURL()
         → pushData(idx, {img_data: dataURL})
         → POST /push_data_blocking_

Python: pop_data_a(idx)   ←  asyncio.Queue.get()
        → base64 decode   →  PIL.Image.open()
```

### Requirements

- Browser must support `navigator.mediaDevices.getDisplayMedia` (Chrome, Edge, Firefox)
- User must grant screen sharing permission when prompted
- The shared screen/window stays active until the browser tab is closed or sharing is stopped